# 1. Setup & Installations\n

In [ ]:
!pip -q install "numpy<2.1" "scipy>=1.13.0" "pandas>=2.2.2" "tensorflow>=2.16" "protobuf<6.0.0"
!pip -q install -U scikit-learn matplotlib seaborn nltk wordcloud joblib gradio lime beautifulsoup4 lxml transformers accelerate

print("Installation completed with compatible versions.")\n

# 2. Imports & Configuration\n

In [ ]:
import os
import re
import gc
import json
import math
import pickle
import random
import warnings
import joblib
import numpy as np
import pandas as pd

from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.sparse import hstack

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score,
    log_loss, brier_score_loss
)
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras import Model, Input
from tensorflow.keras.layers import (
    Dense, Dropout, Conv1D, Embedding, Bidirectional, LSTM, GRU,
    GlobalMaxPooling1D, GlobalAveragePooling1D, SpatialDropout1D,
    LayerNormalization, MultiHeadAttention, Add, Concatenate
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from wordcloud import WordCloud

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

PROJECT_DIR = Path("/content/ISOT_Fake_News_Research_Project")
MODEL_DIR = PROJECT_DIR / "models"
PLOT_DIR = PROJECT_DIR / "plots"
REPORT_DIR = PROJECT_DIR / "reports"
EXPLAIN_DIR = PROJECT_DIR / "explainability"
PRED_DIR = PROJECT_DIR / "predictions"

for folder in [PROJECT_DIR, MODEL_DIR, PLOT_DIR, REPORT_DIR, EXPLAIN_DIR, PRED_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

FAKE_PATH = "/content/Fake.csv"
TRUE_PATH = "/content/True.csv"

WORD_MAX_FEATURES = 100_000
CHAR_MAX_FEATURES = 60_000
MAX_VOCAB_SIZE = 50_000
MAX_SEQUENCE_LENGTH = 500
EMBEDDING_DIM = 128
DL_EPOCHS = 12
DL_BATCH_SIZE = 64

print("=" * 100)
print("SYSTEM INFORMATION")
print("=" * 100)
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
print("Project:", PROJECT_DIR)\n

# 3. Data Loading & Preprocessing\n

In [ ]:
if not os.path.exists(FAKE_PATH) or not os.path.exists(TRUE_PATH):
    print("\nDataset files not found. Please upload Fake.csv and True.csv.")

fake = pd.read_csv(FAKE_PATH)
real = pd.read_csv(TRUE_PATH)

fake.columns = [c.lower().strip() for c in fake.columns]
real.columns = [c.lower().strip() for c in real.columns]

fake["label"] = 0
real["label"] = 1
fake["label_name"] = "Fake"
real["label_name"] = "Real"

df = pd.concat([fake, real], ignore_index=True)

print("\nFake:", fake.shape)
print("Real:", real.shape)
print("Combined:", df.shape)

for column in ["title", "text", "subject", "date"]:
    if column not in df.columns:
        df[column] = ""
    df[column] = (df[column].fillna("").astype(str))

def clean_text(text):
    text = str(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"&\w+;", " ", text)
    text = re.sub(r"\\[a-zA-Z]+", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["raw_combined_text"] = (df["subject"] + " " + df["title"] + " " + df["text"])
df["clean_text"] = df["raw_combined_text"].apply(clean_text)
df = df[df["clean_text"].str.len() >= 30].copy()
df = df.drop_duplicates(subset=["clean_text"]).reset_index(drop=True)

print("\nDataset after duplicate removal:", df.shape)\n

# 4. Exploratory Data Analysis\n

In [ ]:
df["word_count"] = (df["clean_text"].str.split().str.len())
df["character_count"] = (df["clean_text"].str.len())

plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="label_name")
plt.title("ISOT Fake vs Real News Distribution")
plt.tight_layout()
plt.savefig(PLOT_DIR / "class_distribution.png", dpi=200)
plt.show()

plt.figure(figsize=(11, 6))
sns.histplot(data=df, x="word_count", hue="label_name", bins=80, kde=True)
plt.xlim(0, df["word_count"].quantile(0.99))
plt.title("Article Length Distribution")
plt.tight_layout()
plt.savefig(PLOT_DIR / "article_length.png", dpi=200)
plt.show()

for label, label_name in [(0, "Fake"), (1, "Real")]:
    subset = df[df["label"] == label]
    sample_size = min(5000, len(subset))
    text = " ".join(subset.sample(sample_size, random_state=SEED)["clean_text"])
    cloud = WordCloud(width=1400, height=700, background_color="white", max_words=200).generate(text)
    plt.figure(figsize=(14, 7))
    plt.imshow(cloud, interpolation="bilinear")
    plt.axis("off")
    plt.title(f"{label_name} News Word Cloud")
    plt.tight_layout()
    cloud.to_file(str(PLOT_DIR / f"{label_name.lower()}_wordcloud.png"))
    plt.show()\n

# 5. Feature Engineering (TF-IDF)\n

In [ ]:
X = df["clean_text"]
y = df["label"]

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED)

word_vectorizer = TfidfVectorizer(
    lowercase=True, strip_accents="unicode", sublinear_tf=True,
    ngram_range=(1, 2), min_df=2, max_df=0.98, max_features=WORD_MAX_FEATURES, dtype=np.float32
)
X_train_word = word_vectorizer.fit_transform(X_train)
X_val_word = word_vectorizer.transform(X_val)
X_test_word = word_vectorizer.transform(X_test)

char_vectorizer = TfidfVectorizer(
    analyzer="char", sublinear_tf=True, ngram_range=(3, 5),
    min_df=3, max_features=CHAR_MAX_FEATURES, dtype=np.float32
)
X_train_char = char_vectorizer.fit_transform(X_train)
X_val_char = char_vectorizer.transform(X_val)
X_test_char = char_vectorizer.transform(X_test)

X_train_tfidf = hstack([X_train_word, X_train_char]).tocsr()
X_val_tfidf = hstack([X_val_word, X_val_char]).tocsr()
X_test_tfidf = hstack([X_test_word, X_test_char]).tocsr()

joblib.dump(word_vectorizer, MODEL_DIR / "word_tfidf.joblib")
joblib.dump(char_vectorizer, MODEL_DIR / "char_tfidf.joblib")\n

# 6. Model Evaluation Helpers\n

In [ ]:
def sigmoid(x):
    x = np.clip(x, -50, 50)
    return 1 / (1 + np.exp(-x))

def get_probability(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    score = model.decision_function(X)
    return sigmoid(score)

def evaluate_predictions(y_true, probabilities, threshold=0.5):
    predictions = (probabilities >= threshold).astype(int)
    return {
        "Accuracy": accuracy_score(y_true, predictions),
        "Precision": precision_score(y_true, predictions, zero_division=0),
        "Recall": recall_score(y_true, predictions, zero_division=0),
        "F1": f1_score(y_true, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, probabilities),
        "PR-AUC": average_precision_score(y_true, probabilities)
    }

def optimize_threshold(y_true, probabilities):
    thresholds = np.linspace(0.10, 0.90, 161)
    best_threshold, best_f1 = 0.5, -1
    for threshold in thresholds:
        pred = (probabilities >= threshold).astype(int)
        score = f1_score(y_true, pred, zero_division=0)
        if score > best_f1:
            best_f1, best_threshold = score, threshold
    return float(best_threshold), float(best_f1)\n

# 7. Classical Models Training\n

In [ ]:
classical_models = {
    "LogisticRegression": LogisticRegression(solver="lbfgs", max_iter=1000, C=1.0, class_weight="balanced", random_state=SEED, n_jobs=-1),
    "SGDClassifier": SGDClassifier(loss="modified_huber", max_iter=1000, tol=1e-3, class_weight="balanced", random_state=SEED, n_jobs=-1),
    "LinearSVC": LinearSVC(C=0.1, class_weight="balanced", random_state=SEED, max_iter=2000)
}

results_val = {}
probabilities_val = {}

for name, model in classical_models.items():
    model.fit(X_train_tfidf, y_train)
    probs = get_probability(model, X_val_tfidf)
    probabilities_val[name] = probs
    thresh, best_f1 = optimize_threshold(y_val, probs)
    metrics = evaluate_predictions(y_val, probs, threshold=thresh)
    results_val[name] = metrics
    joblib.dump(model, MODEL_DIR / f"{name.lower()}_model.joblib")

results_df = pd.DataFrame(results_val).T
display(results_df.sort_values("F1", ascending=False))\n

# 8. Deep Learning Preparation\n

In [ ]:
tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=MAX_SEQUENCE_LENGTH, padding="post", truncating="post")
X_val_seq = pad_sequences(tokenizer.texts_to_sequences(X_val), maxlen=MAX_SEQUENCE_LENGTH, padding="post", truncating="post")
X_test_seq = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=MAX_SEQUENCE_LENGTH, padding="post", truncating="post")

with open(MODEL_DIR / "tokenizer.pickle", "wb") as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

def build_lstm_model(vocab_size, embedding_dim, input_length):
    model = tf.keras.Sequential([
        Embedding(vocab_size, embedding_dim, input_length=input_length),
        SpatialDropout1D(0.3),
        Bidirectional(LSTM(64, return_sequences=True)),
        Bidirectional(LSTM(32)),
        Dense(64, activation="relu"),
        Dropout(0.5),
        Dense(1, activation="sigmoid")
    ])
    model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
    return model

lstm_model = build_lstm_model(MAX_VOCAB_SIZE, EMBEDDING_DIM, MAX_SEQUENCE_LENGTH)
lstm_model.summary()\n

# 9. LSTM Training\n

In [ ]:
callbacks = [
    EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    ModelCheckpoint(MODEL_DIR / "best_lstm_model.keras", monitor="val_loss", save_best_only=True)
]

history = lstm_model.fit(
    X_train_seq, y_train,
    validation_data=(X_val_seq, y_val),
    epochs=DL_EPOCHS,
    batch_size=DL_BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Train Accuracy")
plt.plot(history.history["val_accuracy"], label="Val Accuracy")
plt.title("LSTM Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Val Loss")
plt.title("LSTM Loss")
plt.legend()
plt.show()\n